## Step1: dataset loading

In [ ]:
!PYTHONPATH=../../src python3 ../../src/datasets/main.py \
    dataset=imagenet \
    forget=instance \
    dataset.init_path="imagenet_example_data" \
    dataset.save_dir="imagenet_example_split" \
    dataset.val_ratio=0.1 \
    forget.forget_idx=[0]


## Step2: model loading

In [ ]:
!PYTHONPATH=../../src python3 -m train.main \
    dataset=imagenet \
    model=resnet18 \
    wandb=default \
    train_cfg=pretrained \
    dataset.load_dir="imagenet_example_split" \
    model.pretrained=True \
    model.num_classes=1000 \
    model.save_dir="artifacts/models"


## Step 3: unlearning 

### neggrad 

In [ ]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=neggrad \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.epochs=1 \
    unlearner.evaluate=false \
    +wandb=default


### scrub

In [ ]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=scrub \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.min_epochs=0 \
    unlearner.cfg.momentum=0 \
    unlearner.cfg.weight_decay=0 \
    unlearner.cfg.max_epochs=1 \
    unlearner.cfg.alpha=1 \
    unlearner.cfg.gamma=0.5 \
    +wandb=default 


## Step 4: GGL Reconstruction

### neggrad

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    data=imagenet \
    reconstructor=ggl \
    original_weights="artifacts/models/unlearn/neggrad/resnet18_42_original.pt" \
    unlearned_weights="artifacts/models/unlearn/neggrad/resnet18_42_unlearned.pt" \
    data.model_name=resnet18 \
    data.labels=[12] \
    data.data_root="imagenet_example_split/forget" \
    reconstructor.cfg.budget=500 \
    reconstructor.lr=0.001 \
    +wandb=default \
    +wandb.extra_config.unlearning_method=neggrad \
    reconstructor.cfg.initial_lr=1 \
    +wandb.extra_config.epochs=1


### scrub

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    data=imagenet \
    reconstructor=ggl \
    original_weights="artifacts/models/unlearn/scrub/resnet18_42_original.pt" \
    unlearned_weights="artifacts/models/unlearn/scrub/resnet18_42_unlearned.pt" \
    data.model_name=resnet18 \
    data.labels=[12] \
    data.data_root="imagenet_example_split/forget" \
    reconstructor.cfg.budget=1000 \
    reconstructor.lr=0.001 \
    reconstructor.cfg.unlearning_method=scrub \
    reconstructor.unlearner.alpha=1 \
    reconstructor.unlearner.gamma=0.5 \
    reconstructor.unlearner.min_epochs=0 \
    reconstructor.unlearner.max_epochs=1 \
    reconstructor.cfg.initial_lr=1 \
    +wandb=default \
    +wandb.extra_config.unlearning_method=scrub \
    +wandb.extra_config.epochs=1


### neggradplus

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    data=imagenet \
    reconstructor=ggl \
    original_weights="artifacts/models/unlearn/neggradplus/resnet18_42_original.pt" \
    unlearned_weights="artifacts/models/unlearn/neggradplus/resnet18_42_unlearned.pt" \
    data.model_name=resnet18 \
    data.labels=[12] \
    data.data_root="imagenet_example_split/forget" \
    reconstructor.cfg.budget=1000 \
    reconstructor.lr=0.001 \
    reconstructor.cfg.unlearning_method=neggradplus \
    reconstructor.unlearner.beta=0.5 \
    +wandbg=default \
    +wandb.extra_config.unlearning_method=neggradplus \
    reconstructor.cfg.initial_lr=1 \
    +wandb.extra_config.epochs=1
